In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [3]:
tech = pd.read_excel("Technology_Skills.xlsx")
skills = pd.read_excel("Skills.xlsx")
activities = pd.read_excel("Work_Activities.xlsx")

In [4]:
tech.columns = tech.columns.str.strip()
skills.columns = skills.columns.str.strip()
activities.columns = activities.columns.str.strip()

In [5]:
print(tech.columns)
print(skills.columns)
print(activities.columns)

Index(['O*NET-SOC Code', 'Title', 'Example', 'Commodity Code',
       'Commodity Title', 'Hot Technology', 'In Demand'],
      dtype='object')
Index(['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID',
       'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound',
       'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date',
       'Domain Source'],
      dtype='object')
Index(['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID',
       'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound',
       'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date',
       'Domain Source'],
      dtype='object')


In [8]:
for s in sorted(skills["Element Name"].unique()):
    print(s)

Active Learning
Active Listening
Complex Problem Solving
Coordination
Critical Thinking
Equipment Maintenance
Equipment Selection
Installation
Instructing
Judgment and Decision Making
Learning Strategies
Management of Financial Resources
Management of Material Resources
Management of Personnel Resources
Mathematics
Monitoring
Negotiation
Operation and Control
Operations Analysis
Operations Monitoring
Persuasion
Programming
Quality Control Analysis
Reading Comprehension
Repairing
Science
Service Orientation
Social Perceptiveness
Speaking
Systems Analysis
Systems Evaluation
Technology Design
Time Management
Troubleshooting
Writing


In [9]:
for a in sorted(activities["Element Name"].unique()):
    print(a)

Analyzing Data or Information
Assisting and Caring for Others
Coaching and Developing Others
Communicating with People Outside the Organization
Communicating with Supervisors, Peers, or Subordinates
Controlling Machines and Processes
Coordinating the Work and Activities of Others
Developing Objectives and Strategies
Developing and Building Teams
Documenting/Recording Information
Drafting, Laying Out, and Specifying Technical Devices, Parts, and Equipment
Establishing and Maintaining Interpersonal Relationships
Estimating the Quantifiable Characteristics of Products, Events, or Information
Evaluating Information to Determine Compliance with Standards
Getting Information
Guiding, Directing, and Motivating Subordinates
Handling and Moving Objects
Identifying Objects, Actions, and Events
Inspecting Equipment, Structures, or Materials
Interpreting the Meaning of Information for Others
Judging the Qualities of Objects, Services, or People
Making Decisions and Solving Problems
Monitoring Proc

In [10]:
tech_skills = [
    "Programming",
    "Systems Analysis",
    "Systems Evaluation",
    "Technology Design",
    "Operations Analysis"
]

In [11]:
tech_activities = [
    "Working with Computers",
    "Processing Information",
    "Analyzing Data or Information",
    "Updating and Using Relevant Knowledge"
]

In [12]:
act_f = activities[
    (activities["Element Name"].isin(tech_activities)) &
    (activities["Scale ID"] == "IM")
]

act_agg = act_f.groupby("O*NET-SOC Code")["Data Value"].mean()
act_agg.name = "tech_activity_score"

In [13]:
skills_f = skills[
    (skills["Element Name"].isin(tech_skills)) &
    (skills["Scale ID"] == "IM")
]

skills_agg = skills_f.groupby("O*NET-SOC Code")["Data Value"].mean()
skills_agg.name = "tech_skill_score"

In [14]:
tech_agg = tech.groupby("O*NET-SOC Code").agg(
    n_tech = ("Example", "nunique"),
    n_hot = ("Hot Technology", lambda x: (x=="Y").sum()),
    n_demand = ("In Demand", lambda x: (x=="Y").sum())
)

In [15]:
df = tech_agg.join(skills_agg).join(act_agg).fillna(0)

In [16]:
from sklearn.preprocessing import StandardScaler

features = ["n_tech", "n_hot", "n_demand", "tech_skill_score", "tech_activity_score"]

scaler = StandardScaler()
X = scaler.fit_transform(df[features])

In [17]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
df["theta"] = pca.fit_transform(X)

In [19]:
df.sort_values("theta", ascending=False).head(10)

,n_tech,n_hot,n_demand,tech_skill_score,tech_activity_score,theta
O*NET-SOC Code,,,,,,
15-1252.00,429,148,33,3.502,4.3225,12.921238
15-1254.00,251,122,41,3.272,4.2100,10.871202
15-1253.00,428,124,22,3.146,4.4850,10.815995
15-1242.00,288,110,32,3.024,4.1075,9.731097
15-1299.08,265,129,25,3.502,4.1600,9.633033
15-1251.00,294,103,19,3.424,4.5550,8.614914
15-1243.00,324,112,13,3.300,4.5000,8.454726
15-1211.00,313,119,11,3.326,4.4950,8.342629
15-1299.09,334,121,9,2.800,4.3050,7.985141


In [20]:
import numpy as np

df["n_tech_log"] = np.log1p(df["n_tech"])
df["n_hot_log"] = np.log1p(df["n_hot"])
df["n_demand_log"] = np.log1p(df["n_demand"])

In [21]:
features = [
    "n_tech_log",
    "n_hot_log",
    "n_demand_log",
    "tech_skill_score",
    "tech_activity_score"
]

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X = scaler.fit_transform(df[features])

pca = PCA(n_components=1)
df["theta_refined"] = pca.fit_transform(X)

In [23]:
df.sort_values("theta_refined", ascending=False).head(10)

,n_tech,n_hot,n_demand,tech_skill_score,tech_activity_score,theta,n_tech_log,n_hot_log,n_demand_log,theta_refined
O*NET-SOC Code,,,,,,,,,,
15-1252.00,429,148,33,3.502,4.3225,12.921238,6.063785,5.003946,3.526361,5.898345
15-1253.00,428,124,22,3.146,4.4850,10.815995,6.061457,4.828314,3.135494,5.419300
15-1254.00,251,122,41,3.272,4.2100,10.871202,5.529429,4.812184,3.737670,5.382759
15-1299.08,265,129,25,3.502,4.1600,9.633033,5.583496,4.867534,3.258097,5.360005
15-1251.00,294,103,19,3.424,4.5550,8.614914,5.686975,4.644391,2.995732,5.277055
15-1243.00,324,112,13,3.300,4.5000,8.454726,5.783825,4.727388,2.639057,5.092166
15-1242.00,288,110,32,3.024,4.1075,9.731097,5.666427,4.709530,3.496508,5.054216
15-1211.00,313,119,11,3.326,4.4950,8.342629,5.749393,4.787492,2.484907,5.051905
15-1221.00,163,73,30,3.446,4.6050,7.516498,5.099866,4.304065,3.433987,5.010299


In [24]:
df[features].corr()

,n_tech_log,n_hot_log,n_demand_log,tech_skill_score,tech_activity_score
n_tech_log,1.000000,0.858531,0.695078,0.471898,0.402817
n_hot_log,0.858531,1.000000,0.712887,0.436740,0.346099
n_demand_log,0.695078,0.712887,1.000000,0.359294,0.320694
tech_skill_score,0.471898,0.436740,0.359294,1.000000,0.834268
tech_activity_score,0.402817,0.346099,0.320694,0.834268,1.000000


In [25]:
print(pca.components_)

[[0.49155643 0.48195159 0.43956308 0.42330708 0.39203362]]


In [26]:
# índice 1: infraestrutura tecnológica
infra_features = ["n_tech_log","n_hot_log","n_demand_log"]

# índice 2: uso tecnológico no trabalho
use_features = ["tech_skill_score","tech_activity_score"]

In [27]:
pca1 = PCA(n_components=1)
df["theta_infra"] = pca1.fit_transform(StandardScaler().fit_transform(df[infra_features]))

pca2 = PCA(n_components=1)
df["theta_use"] = pca2.fit_transform(StandardScaler().fit_transform(df[use_features]))

In [28]:
df["theta_infra"].corr(df["theta_use"])

0.44547193675055247

In [32]:
df.head(10)

,n_tech,n_hot,n_demand,tech_skill_score,tech_activity_score,theta,n_tech_log,n_hot_log,n_demand_log,theta_refined,theta_infra,theta_use
O*NET-SOC Code,,,,,,,,,,,,
11-1011.00,49,16,4,2.998,4.3300,1.334697,3.912023,2.833213,1.609438,2.186488,1.400671,1.791124
11-1011.03,21,14,4,2.880,4.1575,0.824311,3.091042,2.708050,1.609438,1.510089,0.782847,1.509491
11-1021.00,146,49,3,2.348,4.0950,2.741362,4.990433,3.912023,1.386294,2.714034,2.739464,0.811873
11-1031.00,32,11,0,0.000,0.0000,-3.098548,3.496508,2.484907,0.000000,-3.117106,-0.081862,-5.322120
11-2011.00,73,27,6,2.498,3.7900,1.600223,4.304065,3.332205,1.945910,2.247285,2.214659,0.750497
11-2021.00,153,67,6,2.752,3.8350,3.859641,5.036953,4.219508,1.945910,3.366829,3.326732,1.095697
11-2022.00,103,39,5,2.596,4.0100,2.327137,4.644391,3.688879,1.791759,2.734558,2.598736,1.045779
11-2032.00,66,29,5,0.000,0.0000,-1.645848,4.204693,3.401197,1.791759,-1.340582,2.108371,-5.322120
11-2033.00,61,24,8,2.350,3.9400,1.552279,4.127134,3.218876,2.197225,2.164840,2.169610,0.690320
